In [1]:
import pandas as pd
import joblib
import re
from transformers import pipeline
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score,precision_score,recall_score,f1_score,ConfusionMatrixDisplay

In [2]:
imdb = pd.read_csv("/Users/sanasalim/Downloads/Linkific/Day 9/IMDB Dataset.csv")
day9_model = joblib.load('/Users/sanasalim/Downloads/Linkific/Day 9/imdb_sentiment_model.pkl')
day9_vectorizer = joblib.load('/Users/sanasalim/Downloads/Linkific/Day 9/imdb_vectorizer.pkl')

In [3]:
lemmatizer = WordNetLemmatizer()
def clean_text(text):
    # make everything lowercase
    text = text.lower()
    # remove trailing and leading whitespace
    text = text.strip()
    # remove extra whitespace
    text = re.sub(r"\s+"," ",text)
    # remove html tags
    text = re.sub("<.*?>", "", text) 
    # remove special characters
    text = re.sub("[^a-zA-Z]", " ", text)   
    # tokenise  
    tokens = word_tokenize(text)
    # remove stopwords
    sw = set(stopwords.words('english'))
    removed = []
    for word in tokens:
        if word not in sw:
            removed.append(word)
    # lemmatize
    lemmatized = []
    for word in removed:
        lemmatized.append(lemmatizer.lemmatize(word,pos="v"))
    tokens = " ".join(lemmatized)
    return tokens
imdb["sentiment"] = imdb["sentiment"].map({
    "negative": 0,
    "positive": 1})
imdb["cleaned"] = imdb["review"].apply(clean_text)

In [4]:
X = day9_vectorizer.fit_transform(imdb["cleaned"])
y = imdb["sentiment"]
raw_reviews = imdb["review"] 
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.8,random_state=42,stratify=y)

In [5]:
day9_model.fit(X_train, y_train)
y_pred = day9_model.predict(X_test)

In [6]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
precision = precision_score(y_test, y_pred)
print("Precision:",precision.round(4))
recall = recall_score(y_test, y_pred)
print("Recall:",recall.round(4))
f1 = f1_score(y_test, y_pred)
print("f1 score:",f1.round(4))
cv_scores = cross_val_score(day9_model, X, y, cv=5, scoring='accuracy')
print(cv_scores)
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

Accuracy: 0.8872
Precision: 0.8775
Recall: 0.9
f1 score: 0.8886
[0.8855 0.8833 0.8817 0.8814 0.882 ]
              precision    recall  f1-score   support

    Negative       0.90      0.87      0.89      5000
    Positive       0.88      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [7]:
subset_size = 1000
imdb_subset = imdb.sample(n=subset_size,random_state=42)
sentiment_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english",truncation=True,max_length=512)   
predictions = []
count = 0
for review in imdb_subset["review"]:
    count += 1
    if count % 100 == 0:
        print(count)
    sentiment = sentiment_model(review, truncation=True)[0]
    predictions.append(sentiment)

100
200
300
400
500
600
700
800
900
1000


In [11]:
pred_labels = []
for pred in predictions:
    pred_labels.append(
        1 if pred["label"] == "POSITIVE" else 0)

In [10]:
y_true = imdb_subset["sentiment"]
print(y_true.head())

print(predictions[:5])

print(type(predictions[0]))

33553    1
9427     1
199      0
12447    1
39489    0
Name: sentiment, dtype: int64
[{'label': 'NEGATIVE', 'score': 0.9849367737770081}, {'label': 'NEGATIVE', 'score': 0.7579250335693359}, {'label': 'NEGATIVE', 'score': 0.9982203841209412}, {'label': 'POSITIVE', 'score': 0.9992192983627319}, {'label': 'NEGATIVE', 'score': 0.9968709349632263}]
<class 'dict'>


In [13]:
y_true = imdb_subset["sentiment"]
accuracy = accuracy_score(y_true,pred_labels)
precision = precision_score(y_true,pred_labels)
recall = recall_score(y_true,pred_labels)
f1 = f1_score(y_true,pred_labels)
print("Accuracy:", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

Accuracy: 0.865
Precision: 0.8902
Recall: 0.8172
F1 Score: 0.8521
